# ShopStream at Scale — Distributed Batch Analytics with PySpark

**Как запускать:** `Run All` в VS Code, или из терминала `python run_a2.py --scale medium --seed <ID4>`.
Все таблицы и графики сохраняются в папку `results/`.

Части: **A** ингест и хранение · **B** качество данных · **C** аналитика · **D** joins и skew · **E** бенчмарки · **F** отчёт

## 0. Параметры и данные

In [1]:
import os, sys, subprocess

# scale и seed можно передать снаружи (это делает run_a2.py); иначе берутся значения по умолчанию
SCALE = os.environ.get("A2_SCALE", "small")        # разработка: small, финальные цифры: medium
SEED  = int(os.environ.get("A2_SEED", "5820"))     # <-- ЗАМЕНИ 1234 на последние 4 цифры студенческого

DATA    = f"data/shopstream_{SCALE}_{SEED}"
EVENTS  = f"{DATA}/events"
RESULTS = "results"
os.makedirs(RESULTS, exist_ok=True)
print(SCALE, SEED, DATA)

small 5820 data/shopstream_small_5820


In [2]:
# Генерируем данные, только если их ещё нет (тот же seed -> точно такие же данные)
if not os.path.exists(f"{DATA}/_meta.json"):
    subprocess.run([sys.executable, "shopstream_gen.py", "--out", DATA,
                    "--scale", SCALE, "--seed", str(SEED)], check=True)
print(open(f"{DATA}/_meta.json").read())

{
  "scale": "small",
  "seed": 5820,
  "n_users": 8000,
  "n_products": 1500,
  "n_days": 45,
  "n_events_written": 550883,
  "n_files": 180,
  "start_date": "2026-07-01"
}


In [3]:
import os, sys, shutil
print("Python:", sys.executable)
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("java в PATH:", shutil.which("java"))
print("java в conda-окружении:", os.path.exists(os.path.join(sys.prefix, "Library", "bin", "java.exe")))

Python: c:\Users\Dana\miniconda3\envs\shopstream\python.exe
JAVA_HOME: None
java в PATH: None
java в conda-окружении: True


## 1. Spark и окружение

In [4]:
import time, statistics, platform, json
import pandas as pd
import psutil

# Spark должен использовать тот же Python, что и ноутбук
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

if platform.system() == "Windows":
    # Java из conda-окружения: VS Code не активирует conda, поэтому указываем путь сами
    conda_java = os.path.join(sys.prefix, "Library")
    if os.path.exists(os.path.join(conda_java, "bin", "java.exe")):
        os.environ["JAVA_HOME"] = conda_java
        os.environ["PATH"] = os.path.join(conda_java, "bin") + ";" + os.environ["PATH"]
    # winutils для записи файлов
    os.environ["HADOOP_HOME"] = r"C:\hadoop"
    os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

print("JAVA_HOME =", os.environ.get("JAVA_HOME"))

from pyspark.sql import SparkSession, functions as F, types as T

RAM_GB = psutil.virtual_memory().total / 1e9

spark = (SparkSession.builder
         .master("local[*]")
         .appName("ShopStream-A2")
         .config("spark.driver.memory", f"{max(2, int(RAM_GB * 0.5))}g")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.local.dir", os.path.abspath("spark_tmp"))
         .config("spark.ui.port", "4040")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

print("Spark", spark.version, "| ядер:", spark.sparkContext.defaultParallelism)
print("Spark UI: http://localhost:4040")

JAVA_HOME = c:\Users\Dana\miniconda3\envs\shopstream\Library
Spark 3.5.3 | ядер: 16
Spark UI: http://localhost:4040


In [5]:
# Окружение — обязательно для отчёта (все замеры без него не засчитываются)
java_out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
ENV = {
    "OS": platform.platform(),
    "CPU": platform.processor(),
    "CPU cores (logical)": os.cpu_count(),
    "RAM_GB": round(RAM_GB, 1),
    "Python": sys.version.split()[0],
    "Java": next(l for l in java_out.splitlines() if "version" in l).strip(),
    "Spark": spark.version,
    "scale": SCALE, "seed": SEED,
}
json.dump(ENV, open(f"{RESULTS}/environment.json", "w"), indent=2)
ENV

{'OS': 'Windows-10-10.0.26200-SP0',
 'CPU': 'AMD64 Family 26 Model 96 Stepping 0, AuthenticAMD',
 'CPU cores (logical)': 16,
 'RAM_GB': 16.2,
 'Python': '3.11.17',
 'Java': 'openjdk version "17.0.18" 2026-01-20 LTS',
 'Spark': '3.5.3',
 'scale': 'small',
 'seed': 5820}

## 2. Инструменты для замеров
- `bench` — медиана из 3 запусков + 1 прогревочный (не учитывается).
- `force` — заставляет Spark посчитать весь DataFrame, но ничего не сохранять (`noop`).
  `.show()` считает только 20 строк, `.collect()` меряет перенос данных в драйвер — оба врут.

In [6]:
def bench(fn, runs=3, warmup=True):
    """Запускает fn() runs раз. Возвращает (медиана, список времён) в секундах."""
    if warmup:
        fn()                           # прогрев: кэш диска ОС, загрузка классов Java
    times = []
    for _ in range(runs):
        spark.catalog.clearCache()     # чистим кэш Spark между запусками
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return statistics.median(times), times


def force(df):
    """Полностью вычисляет df, ничего не записывая на диск."""
    df.write.format("noop").mode("overwrite").save()

# Part A — Ingestion and storage layout
## A1. Чтение схемы: автоопределение vs явная схема

In [7]:
# Первый взгляд на сырые события
peek = spark.read.json(EVENTS)
peek.printSchema()
peek.show(5, truncate=False)

root
 |-- app_version: string (nullable = true)
 |-- device: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_ts: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- ingest_ts: string (nullable = true)
 |-- price: double (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- session_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- date: date (nullable = true)

+-----------+-------+------------------+--------------------+----------------+--------------------+------+----------+--------+-----------+--------+----------+
|app_version|device |event_id          |event_ts            |event_type      |ingest_ts           |price |product_id|quantity|session_id |user_id |date      |
+-----------+-------+------------------+--------------------+----------------+--------------------+------+----------+--------+-----------+--------+----------+
|5.2.0      |desktop|e_78b7a164cead5004|2

Колонка `date` — не из JSON, а из названия папки `date=...`.
Это дата **прихода события на сервер** (ingest), а не дата события. Пригодится в Part B.

In [8]:
event_schema = T.StructType([
    T.StructField("event_id",    T.StringType()),
    T.StructField("user_id",     T.StringType()),
    T.StructField("session_id",  T.StringType()),
    T.StructField("product_id",  T.StringType()),
    T.StructField("event_type",  T.StringType()),
    T.StructField("event_ts",    T.StringType()),   # строкой: бывает мусор, парсим сами в Part B
    T.StructField("ingest_ts",   T.StringType()),
    T.StructField("price",       T.DoubleType()),
    T.StructField("quantity",    T.LongType()),
    T.StructField("device",      T.StringType()),
    T.StructField("app_version", T.StringType()),
])

In [9]:
def only_read_inferred():  return spark.read.json(EVENTS)
def only_read_explicit():  return spark.read.schema(event_schema).json(EVENTS)
def read_count_inferred(): return spark.read.json(EVENTS).count()
def read_count_explicit(): return spark.read.schema(event_schema).json(EVENTS).count()

rows = []
for name, fn in [("inferred: read only",    only_read_inferred),
                 ("explicit: read only",    only_read_explicit),
                 ("inferred: read + count", read_count_inferred),
                 ("explicit: read + count", read_count_explicit)]:
    med, ts = bench(fn)
    rows.append({"variant": name, "median_s": round(med, 3), "runs_s": [round(t, 3) for t in ts]})

N_RAW = read_count_explicit()
assert N_RAW == read_count_inferred(), "количество строк должно совпадать"

a1 = pd.DataFrame(rows)
a1.to_csv(f"{RESULTS}/A1_schema_read_times.csv", index=False)
print("n_raw =", N_RAW)
a1

n_raw = 550883


,variant,median_s,runs_s
0,inferred: read only,0.578,"[1.347, 0.556, 0.578]"
1,explicit: read only,0.118,"[0.108, 0.118, 0.121]"
2,inferred: read + count,0.923,"[0.95, 0.923, 0.831]"
3,explicit: read + count,0.321,"[0.321, 0.318, 0.322]"


**Вывод для отчёта (дополнить своими цифрами):**
у JSON нет встроенной схемы, поэтому без неё Spark делает **лишний полный проход** по всем файлам,
чтобы угадать типы (видно по строке `inferred: read only` — время тратится ещё до `count()`).
С явной схемой проход один, а типы стабильны и не зависят от случайного мусора в данных.